# Reflection

The Reflection pattern enables an agent to evaluate and iteratively improve its own outputs. Rather than generating a single response and stopping, the agent enters a **Producer-Critic loop**: a _producer_ generates an initial output, a _critic_ evaluates it against the original requirements, and the producer refines its output based on that critique. The loop continues until the critic is satisfied or a maximum iteration count is reached.

This pattern is powerful because critique is often easier than generation — the critic can catch mistakes, omissions, and style violations that the producer missed in its first pass. Separating the producer and critic roles (even using the same underlying model) prevents self-serving rationalization and produces higher-quality results.

**Use cases:** code generation with review, essay drafting with editorial critique, data validation with automatic correction, structured report generation.

## Implementation with Flyte v2

This notebook reimplements the Producer-Critic reflection loop using **Flyte v2 primitives only** — no LangChain, no LangGraph. The loop runs as a single Flyte task with full observability: every LLM call is a traced checkpoint, and each iteration is streamed live to the Flyte UI as an HTML report.

#### LangChain vs Flyte v2 — Key Differences

| Aspect | LangChain / LangGraph | Flyte v2 |
|--------|----------------------|----------|
| **State management** | In-process `MessageHistory` objects | Typed `ConversationMemory` dataclass (serializable, durable) |
| **Checkpointing** | None (restart from scratch on failure) | `@flyte.trace` per LLM call — resumes from last checkpoint on retry |
| **Observability** | Log parsing | Live HTML report in Flyte UI (`flyte.report`) |
| **Warm containers** | N/A | `ReusePolicy` eliminates cold-start cost per loop iteration |
| **Secrets** | `.env` / `os.environ` | `flyte.Secret` injected by cluster (no plaintext in code) |
| **Execution** | In-process only | Local or remote (containers on Kubernetes) |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic

### 2. Export your API key

In [ ]:
%env ANTHROPIC_API_KEY=sk-ant-...

### 3. Import dependencies and configure the Flyte TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import timedelta

import anthropic
import flyte
import flyte.report

flyte.init(
    endpoint="<your-union-endpoint-url>",
    org="<your-union-org>",
    project="<your-project>",
    domain="development",
    image_builder="remote",
    auth_type="DeviceFlow",
)

# The TaskEnvironment bundles image, resources, secrets, and container reuse policy.
# ReusePolicy keeps containers warm between loop iterations — the cold-start cost
# is paid only once for the first LLM call; all subsequent producer + critic calls
# reuse the same warm pod. For a 5-iteration loop this eliminates ~4 cold starts.
_image = (
    flyte.Image.from_debian_base(name="reflection-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.40.0")
)

reflection_env = flyte.TaskEnvironment(
    name="reflection_llm",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        # Create the secret once: flyte create secret ANTHROPIC_API_KEY --value sk-ant-...
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
    reusable=flyte.ReusePolicy(
        replicas=(1, 4),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=10),
    ),
)

### 4. Define the data models

The key design choice: conversation history is a **typed, serializable dataclass** rather than an in-process object. This means Flyte can:
- Store it durably in object storage between task retries
- Display it as structured output in the UI
- Let any downstream task inspect the full producer/critic dialogue

The immutability pattern (`.append()` returns a new object) matches Flyte's data-lineage model — each iteration's memory snapshot is distinct and traceable.

In [ ]:
@dataclass
class Message:
    """A single dialogue turn in the conversation history."""
    role: str       # "user" | "assistant"
    content: str


@dataclass
class ConversationMemory:
    """Immutable-style conversation history passed between Flyte task invocations."""
    messages: list[Message] = field(default_factory=list)

    def append(self, role: str, content: str) -> ConversationMemory:
        """Return a *new* ConversationMemory with the message appended."""
        return ConversationMemory(messages=self.messages + [Message(role, content)])

    def to_api_format(self) -> list[dict]:
        """Convert to the Anthropic messages API format."""
        return [{"role": m.role, "content": m.content} for m in self.messages]


@dataclass
class ReflectionResult:
    """Final typed output of the reflection workflow."""
    final_code: str
    iterations_used: int
    converged: bool      # True when critic approved with CODE_IS_PERFECT
    history: ConversationMemory

### 5. Define system prompts and traced LLM helpers

The `@flyte.trace` decorator turns each LLM call into a **named checkpoint** visible in the Flyte UI.

**Why traces matter for resilience:** Without traces, a pod crash mid-loop restarts the entire task from iteration 0, wasting all previously spent tokens. With `@flyte.trace`, each successful LLM call is checkpointed. On retry, execution resumes from the last completed checkpoint — reducing wasted cost proportional to loop depth.

Note that the **critic receives only `(task_prompt, code)`** — not the producer's full history. This preserves the separation-of-concerns design: the critic evaluates output objectively, without bias from the producer's reasoning context.

In [ ]:
PRODUCER_SYSTEM = """\
You are an expert Python developer.
Produce clean, well-documented, idiomatic Python code.
Return ONLY the raw Python source — no prose, no markdown fences."""

CRITIC_SYSTEM = """\
You are a senior software engineer conducting a meticulous code review.
Evaluate the submitted code strictly against the stated task requirements.
Check for: correctness, edge case coverage, docstring completeness,
PEP 8 compliance, and robust error handling.

If the code satisfies ALL requirements with zero issues, respond with exactly:
  CODE_IS_PERFECT

Otherwise, respond ONLY with a concise bulleted list of actionable critique
points. Do not rewrite the code — critique only."""


@flyte.trace
async def _produce(memory: ConversationMemory, iteration: int) -> str:
    """Traced producer call. Iteration 0 → initial generation; N → refinement."""
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    messages = memory.to_api_format()
    if iteration > 0:
        messages.append({
            "role": "user",
            "content": "Refine the code to fully address all critique points listed above.",
        })
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4096,
        system=PRODUCER_SYSTEM,
        messages=messages,
    )
    return response.content[0].text


@flyte.trace
async def _critique(task_prompt: str, code: str) -> str:
    """Traced critic call. Evaluates code independently of producer history."""
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2048,
        system=CRITIC_SYSTEM,
        messages=[{
            "role": "user",
            "content": f"Original task requirements:\n{task_prompt}\n\nCode to review:\n{code}",
        }],
    )
    return response.content[0].text

### 6. Define the reflection task

The main task orchestrates the Producer-Critic loop. Key decisions:

- **`cache=flyte.Cache(behavior="disable")`** — LLM outputs are non-deterministic; caching would return stale results across runs.
- **`report=True`** — enables the HTML report tab in the Flyte UI, updated live each iteration so you can watch the loop converge in real time.
- **`retries=3, timeout=timedelta(minutes=30)`** — transient API failures are retried automatically; long-running loops won't be killed prematurely.

In [ ]:
def _html_escape(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _render_report(sections: list[str], completed: int, latest_status: str) -> str:
    return f"""
<!DOCTYPE html><html><head><style>
  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', monospace;
         padding: 1.5em; max-width: 900px; margin: auto; }}
  h1   {{ border-bottom: 2px solid #333; padding-bottom: .4em; }}
  pre  {{ white-space: pre-wrap; word-break: break-word; font-size: .85em; }}
  hr   {{ border: none; border-top: 1px solid #ddd; margin: 2em 0; }}
</style></head><body>
  <h1>Reflection Pattern — Live Execution Report</h1>
  <p>Completed iterations: <strong>{completed}</strong> &nbsp;|&nbsp;
     Latest status: <strong>{latest_status}</strong></p>
  {chr(10).join(sections)}
</body></html>
"""


@reflection_env.task(
    retries=3,
    timeout=timedelta(minutes=30),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def reflection_task(
    task_prompt: str,
    max_iterations: int = 3,
) -> ReflectionResult:
    """
    Producer-Critic reflection loop, Flyte v2 native.

    Memory flow:
      1. Seed ConversationMemory with the task prompt as the first user turn.
      2. Producer reads full memory → generates code → appended as assistant turn.
      3. Critic reads (task_prompt, code) independently → returns critique or CODE_IS_PERFECT.
      4. If imperfect, critique is appended to memory as a user turn.
      5. Next iteration: producer sees prompt → code → critique → refinement instruction.
      6. Repeat until CODE_IS_PERFECT or max_iterations reached.
    """
    memory = ConversationMemory().append("user", task_prompt)
    current_code = ""
    converged = False
    report_sections: list[str] = []
    final_iteration = 0

    for i in range(max_iterations):
        final_iteration = i
        label = f"Iteration {i + 1} / {max_iterations}"

        # Producer step
        current_code = await _produce(memory=memory, iteration=i)
        if i > 0:
            memory = memory.append("user", "Refine the code to fully address all critique points listed above.")
        memory = memory.append("assistant", current_code)

        # Critic step
        critique = await _critique(task_prompt=task_prompt, code=current_code)
        converged = "CODE_IS_PERFECT" in critique
        status = "APPROVED" if converged else "NEEDS IMPROVEMENT"

        # Live report update
        color = "green" if converged else "orange"
        report_sections.append(
            f"<section>"
            f"<h2>{label} — <span style='color:{color}'>{status}</span></h2>"
            f"<h3>Producer Output</h3>"
            f"<pre style='background:#f4f4f4;padding:1em;border-radius:4px'>"
            f"<code>{_html_escape(current_code)}</code></pre>"
            f"<h3>Critic Feedback</h3>"
            f"<pre style='background:#fff8e1;padding:1em;border-radius:4px'>"
            f"{_html_escape(critique)}</pre>"
            f"</section><hr/>"
        )
        await flyte.report.replace.aio(_render_report(report_sections, i + 1, status))
        await flyte.report.flush.aio()

        if converged:
            break

        memory = memory.append(
            "user",
            f"Critique from senior engineer (iteration {i + 1}):\n{critique}",
        )

    return ReflectionResult(
        final_code=current_code,
        iterations_used=final_iteration + 1,
        converged=converged,
        history=memory,
    )

### 7. Run locally

In [ ]:
FACTORIAL_PROMPT = """\
Create a Python function named `calculate_factorial` that:
1. Accepts a single integer `n` as input.
2. Calculates and returns its factorial (n!).
3. Includes a clear, complete docstring.
4. Handles the edge case: factorial of 0 is 1.
5. Raises ValueError with a descriptive message for negative inputs.
"""

run = flyte.with_runcontext(mode="local").run(
    reflection_task,
    task_prompt=FACTORIAL_PROMPT,
    max_iterations=3,
)
run.wait()
result = run.outputs()[0]
print(f"Converged: {result.converged}")
print(f"Iterations used: {result.iterations_used}")
print("\n" + "=" * 60)
print(result.final_code)

### Running remotely

Running remotely gives you the full Flyte experience: live report tab, per-iteration checkpoints in the UI, and automatic retry from the last successful checkpoint on pod failure.

1. Create the secret on the cluster:

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-ant-...

2. Switch to remote execution:

In [ ]:
if __name__ == "__main__":
    flyte.init_from_config()
    run = flyte.run(
        reflection_task,
        task_prompt=FACTORIAL_PROMPT,
        max_iterations=3,
    )
    run.wait()
    result = run.outputs()[0]
    print(f"Converged: {result.converged}, Iterations: {result.iterations_used}")
    print(result.final_code)

## Scaling the pattern

The `ReusePolicy` in the `TaskEnvironment` is critical for loop performance. Without it, each producer and critic call incurs a full container cold-start (typically 15–60s). With `ReusePolicy`, the cost is paid once; all subsequent calls in the same loop run in the already-warm container.

For production workloads running many parallel reflection loops, tune `replicas` to match your expected concurrency:

In [ ]:
# Production tuning: run up to 8 concurrent reflection jobs
# Each pod handles 4 async LLM calls simultaneously (producer + critic × 2 pipelines)
production_env = flyte.TaskEnvironment(
    name="reflection_prod",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
    reusable=flyte.ReusePolicy(
        replicas=(2, 8),        # min 2, scale up to 8 pods
        concurrency=4,          # 4 async tasks per pod
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=15),
    ),
)